# 04. LLM Comparison in the Same RAG Pipeline

This notebook compares generators while holding retrieval fixed.

That is important because otherwise it is easy to confuse:
- a better retriever
- a better prompt
- a better generator

Here we will retrieve the same context first, then pass the same evidence into multiple open models.


The workshop treats the system as retrieval-first:
- tokenization defines what the model sees
- retrieval quality often controls answer quality
- generators cannot reliably compensate for weak retrieval
- evaluation should include groundedness, citation, abstention, bilingual behavior, and community-review placeholders

Current notebook: `04_llm_comparison_in_same_rag_pipeline.ipynb`

This notebook isolates the generator by holding retrieval fixed, making it easier to compare groundedness, style, hallucination tendencies, and citation behavior across models.

This is a core workshop notebook.

Workshop sequence:
1. `01_tokenization_playground.ipynb`
2. `02_embeddings_and_similarity.ipynb`
3. `03_retriever_benchmarking_for_rag.ipynb`
4. `04_llm_comparison_in_same_rag_pipeline.ipynb`
5. `05_evaluating_rag_systems.ipynb`
6. `06_optional_lora_or_domain_adaptation.ipynb` (optional)


## Quick concept refresher

- Tokenization turns text into units that models can process.
- Retrieval chooses which passages are available as evidence.
- The retriever selects context; the generator turns context into an answer.
- Evaluation in archive assistants is multi-layered because the system must be useful, grounded, reviewable, and appropriate.


In [ ]:
# Self-contained runtime setup for this notebook.
# Each notebook includes its own seed and device checks so it can run independently in Colab.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")
print(f"Detected device: {DEVICE}")
print(f"Seed set to: {SEED}")

NOTEBOOK_CONTEXT = {
    "seed": SEED,
    "device": DEVICE,
    "notebook": __name__ if "__name__" in globals() else "notebook",
    "framing": "retrieval-first archive assistant workshop",
}

NOTEBOOK_CONTEXT


In [ ]:
# In Colab, uncomment if needed.

# !pip -q install sentence-transformers transformers accelerate pandas

print("LLM comparison dependencies are ready once the required packages are installed.")


In [ ]:
# Imports.

import re
import textwrap

import pandas as pd
try:
    import torch
except ImportError:
    torch = None
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline


In [ ]:
# A small retrieval corpus for the RAG experiment.

rag_corpus = [
    {
        "chunk_id": "C01",
        "text": "Public summaries can be shown openly, but restricted ceremonial recordings require a community access review before release.",
    },
    {
        "chunk_id": "C02",
        "text": "The bilingual catalog sheet documents place-name spelling variants and recommends citing the original source note.",
    },
    {
        "chunk_id": "C03",
        "text": "Some teaching stories are marked winter-only and should not be surfaced outside the approved seasonal context.",
    },
    {
        "chunk_id": "C04",
        "text": "Transcript quality notes mention merged speaker turns, missing diacritics, and OCR mistakes in interview records.",
    },
    {
        "chunk_id": "C05",
        "text": "Governance notes explain that kinship-based access rules must be checked before showing search results to the public.",
    },
]

rag_df = pd.DataFrame(rag_corpus)
rag_df


In [ ]:
# Build a fixed retriever.
# We use a single dense retriever here so the generator comparison is the only moving part.

retriever_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
retriever_model = SentenceTransformer(retriever_model_name)

rag_embeddings = retriever_model.encode(
    rag_df["text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)


def retrieve_context(query: str, top_k: int = 3):
    query_embedding = retriever_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, rag_embeddings)[0]
    ranked = rag_df.copy()
    ranked["score"] = scores
    return ranked.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


In [ ]:
# Default models.
# Larger placeholders are included as optional comments.

MODEL_SPECS = {
    "flan_t5_small": {
        "model_name": "google/flan-t5-small",
        "task": "text2text-generation",
    },
    "flan_t5_base": {
        "model_name": "google/flan-t5-base",
        "task": "text2text-generation",
    },
    # Optional heavier model ideas for a GPU-backed Colab session:
    # "mistral_7b_instruct": {
    #     "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
    #     "task": "text-generation",
    # },
}

list(MODEL_SPECS.keys())


In [ ]:
# Cache pipelines so models are only loaded once.
# This keeps repeated experimentation manageable.

model_cache = {}
PIPELINE_DEVICE = 0 if (torch is not None and torch.cuda.is_available()) else -1


def get_generation_pipeline(model_key: str):
    spec = MODEL_SPECS[model_key]
    if model_key not in model_cache:
        model_cache[model_key] = pipeline(
            spec["task"],
            model=spec["model_name"],
            tokenizer=spec["model_name"],
            device=PIPELINE_DEVICE,
        )
    return model_cache[model_key]


In [ ]:
# Prompt builder.
# The prompt explicitly asks for grounded answering, citations, and abstention when evidence is weak.

def build_prompt(query: str, retrieved_df: pd.DataFrame) -> str:
    evidence_blocks = []
    for row in retrieved_df.itertuples():
        evidence_blocks.append(f"[{row.chunk_id}] {row.text}")

    evidence_text = "\n".join(evidence_blocks)

    return textwrap.dedent(
        f"""
        You are assisting with a community-governed archive.
        Answer the question using only the evidence below.
        If the evidence is insufficient, say ABSTAIN and explain why briefly.
        Cite chunk ids in square brackets.

        Question: {query}

        Evidence:
        {evidence_text}

        Response format:
        Answer: <grounded answer or ABSTAIN>
        Citations: [chunk ids]
        """
    ).strip()


In [ ]:
# Small robustness fix:
# text2text-generation pipelines often return `generated_text`,
# but keeping a parser helper makes the behavior explicit and easier to modify.

def extract_generated_text(pipeline_output):
    first = pipeline_output[0]
    if "generated_text" in first:
        return first["generated_text"]
    if "summary_text" in first:
        return first["summary_text"]
    return str(first)


def generate_rag_answer(model_key: str, query: str, retrieved_df: pd.DataFrame, max_new_tokens: int = 128):
    generator = get_generation_pipeline(model_key)
    prompt = build_prompt(query, retrieved_df)
    raw_output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return {
        "model_key": model_key,
        "prompt": prompt,
        "response": extract_generated_text(raw_output),
    }


In [ ]:
# Compare models on the same retrieval result.
# Feel free to change the query or top_k.

query = "Which materials require review before they can be released publicly?"
retrieved_context = retrieve_context(query, top_k=3)

print("Retrieved context:")
display(retrieved_context[["chunk_id", "score", "text"]])

model_outputs = []
for model_key in MODEL_SPECS:
    result = generate_rag_answer(model_key, query, retrieved_context)
    model_outputs.append(result)

outputs_df = pd.DataFrame(model_outputs)
outputs_df[["model_key", "response"]]


In [ ]:
# Basic citation inspection.
# This is not a full evaluation pipeline, but it helps surface whether a model is grounding its answer.

def parse_citations(text: str):
    return re.findall(r"\[(C\d+)\]", text)


citation_rows = []
retrieved_ids = set(retrieved_context["chunk_id"].tolist())

for row in outputs_df.itertuples():
    citations = parse_citations(row.response)
    citation_rows.append(
        {
            "model_key": row.model_key,
            "citations_found": citations,
            "all_citations_in_retrieved_context": set(citations).issubset(retrieved_ids),
            "abstained": "ABSTAIN" in row.response.upper(),
        }
    )

citation_df = pd.DataFrame(citation_rows)
citation_df


## Exercises

- Compare a smaller and larger model while keeping retrieval fixed.
- Ask a question that the retrieved evidence does not support. Does the model abstain?
- Inspect whether the model cites the chunks it actually used.
- Identify cases where a more capable generator still cannot rescue weak retrieval.


In [ ]:
# Optional exercise cell:
# try a query that should trigger abstention or expose hallucination tendencies.

exercise_query = "Which donor agreement allows direct quotation of restricted recordings?"
exercise_context = retrieve_context(exercise_query, top_k=3)
display(exercise_context[["chunk_id", "score", "text"]])

exercise_outputs = []
for model_key in MODEL_SPECS:
    exercise_outputs.append(generate_rag_answer(model_key, exercise_query, exercise_context))

pd.DataFrame(exercise_outputs)[["model_key", "response"]]
